In [ ]:
import requests
import pandas as pd
import apimoex
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import datetime as dt
import os
import sklearn
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

## Загрузка котировок ближайшей экспирации

In [ ]:
path = r'data'
df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
print(list(df_ext['isin'].unique()))

stocks = {
    'AFLT': 'AF',
    'ALRS': 'AL',
    'CHMF': 'CH',
    'FEES': 'FS',
    'GAZR': 'GZ',
    'GMKN': 'GK',
    'HYDR': 'HY',
    'LKOH': 'LK',
    'MGNT': 'MN',
    'MOEX': 'ME',
    'MTSI': 'MT',
    'NLMK': 'NM',
    'NOTK': 'NK',
    'ROSN': 'RN',
    'RTKM': 'RT',
    'SBPR': 'SP',
    'SBRF': 'SR',
    'SNGP': 'SG',
    'SNGR': 'SN',
    'TATN': 'TT',
    'TRNF': 'TN',
    'VTBR': 'VB',
    'MAGN': 'MG',
    'PLZL': 'PZ',
    'YNDF': 'YN',
    'AFKS': 'AK',
    'IRAO': 'IR',
    'POLY': 'PO',
    'PIKK': 'PI',
    'SPBE': 'SE',
    'RUAL': 'RL',
    'PHOR': 'PH',
    'DSKY': 'DY',
    'SMLT': 'SS',
    'MTLR': 'MC',
    'RSTI': 'RE',
    'SIBN': 'SO',
    'TCSI': 'TI',
    'FIVE': 'FV',
    'MAIL': 'ML',
    'OZON': 'OZ',
    'SPYF': 'SF',
    'NASD': 'NA',
    'POSI': 'PS',
    'STOX': 'SX',
    'HANG': 'HS',
    'DAX': 'DX',
    'NIKK': 'N2',
    'ISKJ': 'IS',
    'WUSH': 'WU',
    'MVID': 'MV',
    'CBOM': 'CM',
    'SGZH': 'SZ',
    'BELU': 'BE',
    'FLOT': 'FL',
    'BSPB': 'BS',
    'BANE': 'BN',
    'KMAZ': 'KM'
}

commodities = {   
    'BR': 'BR',
    'GOLD': 'GD',
    'GL': 'GL',
    'GLDRUBF': 'GLDRUBF',
    'PLD': 'PD',
    'PLT': 'PT',
    'SILV': 'SV',
    'SUGR': 'SA',
    'SLV': 'SL',
    'ALMN': 'AM',
    'CL': 'CL',
    'Co': 'Co',
    'GLD': 'GO',
    'Nl': 'Nl',
    'Zn': 'Zn',
    'NG': 'NG',
    'WH4': 'WH',
    'WHEAT': 'W4',
    'SUGAR': 'Su'}

indexies = {
    'MIX': 'MX',
    'MXI': 'MM',
    'RTS': 'RI',
    'RTSM': 'RM',
    'RVI': 'VI',
    'HOME': 'HO',
    'OGI': 'OG',
    'MMI': 'MA',
    'FNI': 'FN',
    'CNI': 'CS',
    'RGBI': 'RB',
    'IMOEXF': 'IMOEXF'}

curr = {
    'CNY': 'CR',
    'Eu': 'Eu',
    'Si': 'Si',
    'USDRUBF': 'USDRUBF',
    'EURRUBF': 'EURRUBF',
    'CNYRUBF': 'CNYRUBF',
    'TRY': 'TY',
    'HKD': 'HK',
    'AED': 'AE',
    'INR': 'I2',
    'KZT': 'KZ',
    'AMD': 'AR',
    'ED': 'ED',
    'AUDU': 'AU',
    'GBPU': 'GU',
    'UCAD': 'CA',
    'UCHF': 'CF',
    'UJPY': 'JP',
    'UTRY': 'TR',
    'UCNY': 'UC',
    'ECAD': 'EC',
    'EGBP': 'EG',
    'EJPY': 'EJ'
}

month = {
    1: ['F', '01-01', '01-31'],
    2: ['G', '02-01', '02-28'],
    3: ['H', '03-01', '03-31'],
    4: ['J', '04-01', '04-30'],
    5: ['K', '05-01', '05-31'],
    6: ['M', '06-01', '06-30'],
    7: ['N', '07-01', '07-31'],
    8: ['Q', '08-01', '08-31'],
    9: ['U', '09-01', '09-30'],
    10: ['V', '10-01', '10-31'],
    11: ['X', '11-01', '11-30'],
    12: ['Z', '12-01', '12-31']
}
dict = stocks|indexies|commodities|curr


In [ ]:
def Closest_quots(ISIN_pos,year):
    path = r'data'
    df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')
    df_data = pd.DataFrame()
    df_out = pd.DataFrame()
    expireation_date = '202'+str(year)+'-01-01'
    if ISIN_pos in commodities.keys():
        month_list = [1,2,3,4,5,6,7,8,9,10,11,12]
    else:
        month_list = [3,6,9,12]
    for i in month_list:
        try:
            instument = dict[ISIN_pos]+month[i][0]+str(year) #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx
            with requests.Session() as session:
                data = apimoex.get_market_candles(session=session,security=instument, start = expireation_date, market="forts",engine="futures",interval=24)
                df_data = pd.DataFrame(data)
        
            try:
                df_data=df_data.iloc[:,[0,2]]
                #df_data['close']=df_data.apply(lambda x: x[['close']]/1000, axis = 1)
                df_data['begin'] = df_data.apply(lambda x: (x['begin'])[:10], axis = 1)
                df_data['instrument'] = instument
                #display(df_data)
                df_ext = df_ext[df_ext['isin']==ISIN_pos]
                df_merged = df_data.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()
                expireation_date = df_merged['moment'][len(df_merged['moment'])-1]
                df_out = pd.concat([df_out, df_merged], ignore_index=True)
                
                # df_merged.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
                #df_out.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
                # display(
                # df_merged.head(1),
                # df_merged.tail(1))
            except IndexError:
                pass
        except KeyError:
            pass
    return(df_out)

In [ ]:
best_appr = pd.DataFrame()
path = r'data'
df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')
#В list вставляется словарь с нужной секцией
instr = stocks
List = list(instr.keys())
ISIN_pos = 0
for ISIN_pos in range(len(List)):
    print(List[ISIN_pos])
    df_out = Closest_quots(List[ISIN_pos],3)
    #display(df_out.head(1))
    #del ISIN_pos
    
    #df_merged = df_data.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()
    
    # df_merged.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
    # df_merged.head(1)
    try:
        Y = np.array(df_out['close'].to_list())
        X = np.array(df_out['long_position_jur'] + df_out['long_position_fiz'].to_list())
        X = X.reshape(-1,1)
        Y = Y.reshape(-1,1)
            
        try:
            # reg = LinearRegression().fit(X, Y)
            # print('ISIN: ', ISIN_pos)
            # print('R^2 = ', reg.score(X,Y))
            # print('intercept:', reg.intercept_)
            # print('slope:', reg.coef_)
            offset = 60
            df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
            for i in range(offset):
                Y = np.array(df_out['close'].to_list()).reshape(-1,1)
                X = np.array(df_out['long_position_jur'] + df_out['long_position_fiz'].to_list()).reshape(-1,1)
                
                Y = Y[i:]
                X = X[:len(X)-i]
                reg = LinearRegression().fit(X, Y)
                # print('Offset: ' , i)
                # print('R^2 = ', reg.score(X,Y))
                # print('intercept:', reg.intercept_)
                # print('slope:', reg.coef_)
    
                df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out['isin'][1]] 
            best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
            #display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])
            
            plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
            #plt.ylim(0.7,1)
            plt.xlabel(List[ISIN_pos])
            #plt.show()
            plt.savefig(r'plots/stocks/'+List[ISIN_pos]+'.png')
            plt.clf()
            print(ISIN_pos)
            
        except ValueError:
            print('NA')
    except KeyError:
            print('KeyError')

print(("\033[31m {}" .format('Done')))
best_appr.to_excel(r'data/OUT/best_appr_curr.xlsx')

## Анализ котировок

In [ ]:
#выбор актива
df_out = Closest_quots('MIX',3)

#График
x_titles = np.array(df_out['begin'].to_list())
y_values = np.array(df_out['close'].to_list())

fig,ax = plt.subplots()

plt.ylabel('Стоимость контракта, руб.')

ax.plot(x_titles , y_values , color='b',label=df_out['name'][1])
plt.xticks(x_titles[::10],size = 8,rotation = 45)
plt.legend(loc = 2)


approximator = df_out['long_position_jur']+df_out['long_position_fiz']


x_titles1 = np.array(df_out['begin'].to_list())
y_values1 = np.array(approximator.to_list())

ax2 = ax.twinx ()
ax2.plot(x_titles1, y_values1[:] , color='r', label='Отрытые позиции')

plt.legend(loc = 4)
plt.ylabel("Кол-во контрактов")

fig.set_size_inches(10, 6)
plt.show()

#Регрессии
offset = 60
df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
best_appr = pd.DataFrame()
for i in range(offset):
    Y = np.array(df_out['close'].to_list()).reshape(-1,1)
    X = np.array(approximator.to_list()).reshape(-1,1)
    
    Y = Y[i:]
    X = X[:len(X)-i]
    reg = LinearRegression().fit(X, Y)
    # print('Offset: ' , i)
    # print('R^2 = ', reg.score(X,Y))
    # print('intercept:', reg.intercept_)
    # print('slope:', reg.coef_)
    df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out['isin'][1]] 
best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])

plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
#plt.ylim(0.8,1)
plt.show()


#График со сдвигом
Y = np.array(df_out['close'].to_list()).reshape(-1,1)
X = np.array(approximator.to_list()).reshape(-1,1)

i = df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]][['offset']].iloc[0].iloc[0]

Y = Y[i:]
X = X[:len(X)-i]

x_titles = np.array(df_out['begin'].to_list())[i:]
y_values = Y

fig,ax = plt.subplots()

plt.ylabel('Стоимость контракта, руб.')

ax.plot(x_titles , y_values , color='b',label=df_out['name'][1])
plt.xticks(x_titles[::10],size = 8,rotation = 45)
plt.legend(loc = 2)


approximator = df_out['long_position_jur']+df_out['long_position_fiz']


y_values1 = X

ax2 = ax.twinx ()
ax2.plot(x_titles, y_values1[:] , color='r', label='Отрытые позиции')

plt.legend(loc = 4)
plt.ylabel("Кол-во контрактов")

fig.set_size_inches(10, 6)
plt.show()

In [ ]:
#График
x_titles = np.array(df_out['begin'].to_list())
y_values = np.array(df_out['close'].to_list())

fig,ax = plt.subplots()

plt.ylabel('Стоимость контракта, руб.')

ax.plot(x_titles , y_values , color='b',label=df_out['name'][1])
plt.xticks(x_titles[::10],size = 8,rotation = 45)
plt.legend(loc = 2)


approximator = df_out['long_position_jur']+df_out['long_position_fiz']


x_titles1 = np.array(df_out['begin'].to_list())

y1=np.array(df_out['long_position_jur']/(df_out['long_position_jur']+df_out['long_position_fiz']).to_list())[:]
y2=np.array(df_out['short_position_jur']/(df_out['short_position_jur']+df_out['short_position_fiz']).to_list())[:]
y3=np.array(df_out['long_position_fiz']/(df_out['long_position_jur']+df_out['long_position_fiz']).to_list())[:]
y4=np.array(df_out['short_position_fiz']/(df_out['short_position_jur']+df_out['short_position_fiz']).to_list())[:]

ax2 = ax.twinx ()
ax2.plot(x_titles1, y1 , label='long_pos_share_jur')
# ax2.plot(x_titles1, y2 , label='short_pos_share_jur')
# ax2.plot(x_titles1, y3 , label='long_pos_share_fiz')
# ax2.plot(x_titles1, y4 , label='short_pos_share_fiz')

plt.legend(loc = 4)
plt.ylabel("доли контрактов")

fig.set_size_inches(10, 6)
plt.show()

In [ ]:
df_out.info()

In [ ]:
x_titles

## Лучшие регрессии на ближайшие фьючерсы

In [ ]:
#reference
best_appr = pd.DataFrame()
df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
for j in list(df_ext['isin'].unique()):
    try:
        ISIN_pos = j #From list below
        instument = dict[ISIN_pos]+'Z'+'3' #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx

        with requests.Session() as session:
            data = apimoex.get_market_candles(session=session,security=instument, start = '2023-01-01', market="forts",engine="futures",interval=24)
            df_data = pd.DataFrame(data)
        try:
            df_data=df_data.iloc[:,[0,2]]
            #df_data['close']=df_data.apply(lambda x: x[['close']]/1000, axis = 1)
            df_data['begin'] = df_data.apply(lambda x: (x['begin'])[:10], axis = 1)
            df_data['instrument'] = instument
            #display(df_data)
            path = r'data'
            df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
            df_ext = df_ext[df_ext['isin']==ISIN_pos]
            df_merged = df_data.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()
        except IndexError:
            print('indexerr')
        # df_merged.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
        # df_merged.head(1)
        
        Y = np.array(df_merged['close'].to_list())
        X = np.array(df_merged['long_position_jur'] + df_merged['long_position_fiz'].to_list())
        
        X = X.reshape(-1,1)
        Y = Y.reshape(-1,1)
        
        try:
            # reg = LinearRegression().fit(X, Y)
            # print('ISIN: ', ISIN_pos)
            # print('R^2 = ', reg.score(X,Y))
            # print('intercept:', reg.intercept_)
            # print('slope:', reg.coef_)
            offset = 60
            df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
            for i in range(offset):
                Y = np.array(df_merged['close'].to_list()).reshape(-1,1)
                X = np.array(df_merged['long_position_jur'] + df_merged['long_position_fiz'].to_list()).reshape(-1,1)
                
                Y = Y[i:]
                X = X[:len(X)-i]
                reg = LinearRegression().fit(X, Y)
                # print('Offset: ' , i)
                # print('R^2 = ', reg.score(X,Y))
                # print('intercept:', reg.intercept_)
                # print('slope:', reg.coef_)

                df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_merged['isin'][1]] 
            best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
            display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])
            
            # plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
            # #plt.ylim(0.7,1)
            # plt.xlabel(j)
            # #plt.show()
            # plt.savefig(r'plots/F_reggr/'+j+'.png')
            # plt.clf()
            print(j)
            
        except ValueError:
            print('NA')
    except KeyError:
        print(i)

print(("\033[31m {}" .format('Done')))
best_appr.to_excel(r'data/OUT/best_appr.xlsx')

In [ ]:
def Closest_quots(ISIN_pos,year):
    path = r'data'
    df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
    df_data = pd.DataFrame()
    df_out = pd.DataFrame()
    expireation_date = '202'+str(year)+'-01-01'
    for i in [3,6,9,12]:
        try:
            instument = dict[ISIN_pos]+month[i][0]+str(year) #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx
            with requests.Session() as session:
                data = apimoex.get_market_candles(session=session,security=instument, start = expireation_date, market="forts",engine="futures",interval=24)
                df_data = pd.DataFrame(data)
        
            try:
                df_data=df_data.iloc[:,[0,2]]
                #df_data['close']=df_data.apply(lambda x: x[['close']]/1000, axis = 1)
                df_data['begin'] = df_data.apply(lambda x: (x['begin'])[:10], axis = 1)
                df_data['instrument'] = instument
                #display(df_data)
                df_ext = df_ext[df_ext['isin']==ISIN_pos]
                df_merged = df_data.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()
                expireation_date = df_merged['moment'][len(df_merged['moment'])-1]
                df_out = pd.concat([df_out, df_merged], ignore_index=True)
                
                # df_merged.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
                #df_out.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
                # display(
                # df_merged.head(1),
                # df_merged.tail(1))
            except IndexError:
                pass
        except KeyError:
            pass
    return(df_out)
    
Closest_quots('MIX',3)

In [ ]:
best_appr = pd.DataFrame()
path = r'data'
df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
List = list(df_ext['isin'].unique())
ISIN_pos = 0
for ISIN_pos in range(len(List)):
    print(List[ISIN_pos])
    df_out = Closest_quots(List[ISIN_pos])
    #display(df_out.head(1))
    #del ISIN_pos
    
    #df_merged = df_data.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()
    
    # df_merged.to_excel(path+'/OUT/Pos_change/for_particular_graph.xlsx')
    # df_merged.head(1)
    try:
        Y = np.array(df_out['close'].to_list())
        X = np.array(df_out['long_position_jur'] + df_out['long_position_fiz'].to_list())
        X = X.reshape(-1,1)
        Y = Y.reshape(-1,1)
            
        try:
            # reg = LinearRegression().fit(X, Y)
            # print('ISIN: ', ISIN_pos)
            # print('R^2 = ', reg.score(X,Y))
            # print('intercept:', reg.intercept_)
            # print('slope:', reg.coef_)
            offset = 60
            df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
            for i in range(offset):
                Y = np.array(df_out['close'].to_list()).reshape(-1,1)
                X = np.array(df_out['long_position_jur'] + df_out['long_position_fiz'].to_list()).reshape(-1,1)
                
                Y = Y[i:]
                X = X[:len(X)-i]
                reg = LinearRegression().fit(X, Y)
                # print('Offset: ' , i)
                # print('R^2 = ', reg.score(X,Y))
                # print('intercept:', reg.intercept_)
                # print('slope:', reg.coef_)
    
                df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out['isin'][1]] 
            best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
            #display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])
            
            plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
            #plt.ylim(0.7,1)
            plt.xlabel(ISIN_pos)
            #plt.show()
            plt.savefig(r'plots/F_reggr_closest/'+List[ISIN_pos]+'.png')
            plt.clf()
            print(ISIN_pos)
            
        except ValueError:
            print('NA')
    except KeyError:
            print('KeyError')

print(("\033[31m {}" .format('Done')))
best_appr.to_excel(r'data/OUT/best_appr.xlsx')

In [ ]:
def Closest_quots_only(ISIN_pos, year):
    df_closest_exp = pd.DataFrame()
    expireation_date = '202'+str(year)+'-01-01'
    for i in [3,6,9,12]:
        try:
            instument = dict[ISIN_pos]+month[i][0]+str(year) #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx
            with requests.Session() as session:
                data = apimoex.get_market_candles(session=session,security=instument, start = expireation_date, market="forts",engine="futures",interval=24)
                df_data = pd.DataFrame(data)
                expireation_date = df_data['begin'][len(df_data['begin'])-1]

            try:
                #df_data=df_data.iloc[:,[0,2]]
                #df_data['close']=df_data.apply(lambda x: x[['close']]/1000, axis = 1)
                df_data['begin'] = df_data.apply(lambda x: (x['begin'])[:10], axis = 1)
                df_data['instrument'] = instument
                df_closest_exp = pd.concat([df_closest_exp,df_data], ignore_index=True)
                df_closest_exp.drop_duplicates(subset='begin', keep='first', inplace=True)
            except IndexError:
                pass
        
        except KeyError:
            pass
    return(df_closest_exp)
    

In [ ]:
df_export = Closest_quots_only('MIX',3)

In [ ]:
df_export.to_excel(r'data/OUT/df_export.xlsx')

In [ ]:
df_quotes = pd.DataFrame()
for i in [0,1,2,3]:
    df_closest_exp = Closest_quots_only('MIX',i)
    df_quotes = pd.concat([df_quotes,df_closest_exp], ignore_index=True)


ISIN_pos = 'MIX'


path = r'data'
df_ext = pd.read_excel(path+'/OUT/Pos_change/Change_for_graph.xlsx')
df_ext = df_ext[df_ext['isin']==ISIN_pos]
df_out = df_quotes.merge(df_ext,how='left',left_on='begin',right_on='moment').dropna()

In [ ]:
x_titles = np.array(df_out['begin'].to_list())
y_values = np.array(df_out['close'].to_list())

fig,ax = plt.subplots()

plt.ylabel('Стоимость контракта, руб.')

ax.plot(x_titles , y_values , color='b',label=df_merged['name'][1])
plt.xticks(x_titles[::10],size = 8,rotation = 45)
plt.legend(loc = 2)

approximator = df_out['long_position_jur']+df_out['long_position_fiz']

x_titles1 = np.array(df_out['begin'].to_list())
y_values1 = np.array(approximator.to_list()).reshape(-1,1)

ax2 = ax.twinx ()
ax2.plot(x_titles1, y_values1[:] , color='r', label='Отрытые позиции')

plt.legend(loc = 4)
plt.ylabel("Кол-во контрактов")

fig.set_size_inches(10, 6)
plt.show()

In [ ]:
offset = 60
df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
best_appr = pd.DataFrame()
for i in range(offset):
    Y = np.array(df_out['close'].to_list()).reshape(-1,1)
    X = np.array(approximator.to_list()).reshape(-1,1)
    
    Y = Y[i:]
    X = X[:len(X)-i]
    reg = LinearRegression().fit(X, Y)
    # print('Offset: ' , i)
    # print('R^2 = ', reg.score(X,Y))
    # print('intercept:', reg.intercept_)
    # print('slope:', reg.coef_)
    df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out['isin'][1]] 
best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])
            

In [ ]:
plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
#plt.ylim(0.8,1)
plt.show()

In [ ]:
#Open/close Spread
df_quotes = pd.DataFrame()
for i in [0,1,2,3]:
    df_closest_exp = Closest_quots_only('MIX',i)
    df_quotes = pd.concat([df_quotes,df_closest_exp], ignore_index=True)
df_quotes['spread'] = df_quotes['high']-df_quotes['low']

In [ ]:
df_out.head(1)